# 02 — Exploratory Data Analysis

Descriptive statistics, distribution checks, and all key figures.
Every plot is saved to `output/figures/`.

| | |
|---|---|
| **Inputs** | `data/processed/merged_analysis.csv` |
| **Outputs** | `output/figures/*.png` |

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Paths 
DATA_PROC = Path("../data/processed")
FIG_OUT   = Path("../output/figures")
FIG_OUT.mkdir(parents=True, exist_ok=True)

# Style 
GREEN = "#00693E"   # Dartmouth green
DARK  = "#1a1a2e"
plt.rcParams.update({"figure.dpi": 150, "font.family": "sans-serif"})

# Load data 
df = pd.read_csv(DATA_PROC / "merged_analysis.csv")
print(f"Loaded: {df.shape}")
df.head(3)

In [ ]:
# Descriptive statistics table 
desc_cols = ["re_support", "ideology", "climate_concern",
             "education", "income", "age", "coal_share"]
desc_cols = [c for c in desc_cols if c in df.columns]
desc = df[desc_cols].describe().round(3)
print("Descriptive Statistics")
print(desc)

In [ ]:
# Figure 1: Distribution of RE support (DV) 
fig, ax = plt.subplots(figsize=(7, 4))
counts = df["re_support"].value_counts().sort_index()
bars = ax.bar(counts.index, counts.values, color=GREEN, edgecolor="white", linewidth=0.5)

ax.set_xlabel("RE Support (1 = Strongly Oppose, 5 = Strongly Support)", fontsize=11)
ax.set_ylabel("Number of Respondents", fontsize=11)
ax.set_title("Distribution of Renewable Energy Support", fontsize=13, fontweight="bold")
ax.axvline(df["re_support"].mean(), color=DARK, linestyle="--", linewidth=1.5,
           label=f"Mean = {df['re_support'].mean():.2f}")
ax.legend(fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "re_support_distribution.png")
plt.show()
print("Saved: re_support_distribution.png")

In [ ]:
# Figure 2: Mean RE support by state 
state_means = (
    df.groupby("state")["re_support"]
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(state_means["state"], state_means["re_support"],
        color=GREEN, edgecolor="white", linewidth=0.3)
ax.set_xlabel("Mean RE Support (1–5)", fontsize=11)
ax.set_title("Mean Renewable Energy Support by State", fontsize=13, fontweight="bold")
ax.axvline(df["re_support"].mean(), color=DARK, linestyle="--", linewidth=1.2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "state_support_bar.png")
plt.show()
print("Saved: state_support_bar.png")

In [ ]:
# Figure 3: Coal share vs. RE support scatter 
state_agg = df.groupby("state").agg(
    re_support_mean=("re_support", "mean"),
    coal_share=("coal_share", "first")
).reset_index()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(state_agg["coal_share"], state_agg["re_support_mean"],
           color=GREEN, edgecolors=DARK, linewidths=0.5, s=60, alpha=0.85)

# Add state labels for high-coal states
for _, row in state_agg[state_agg["coal_share"] > 0.3].iterrows():
    ax.annotate(row["state"], (row["coal_share"], row["re_support_mean"]),
                fontsize=7, xytext=(4, 0), textcoords="offset points")

# Trend line
m, b = np.polyfit(state_agg["coal_share"], state_agg["re_support_mean"], 1)
x_line = np.linspace(state_agg["coal_share"].min(), state_agg["coal_share"].max(), 100)
ax.plot(x_line, m * x_line + b, color=DARK, linestyle="--", linewidth=1.2, alpha=0.7)

ax.set_xlabel("State Coal Share (avg. 1990–2023)", fontsize=11)
ax.set_ylabel("Mean RE Support", fontsize=11)
ax.set_title("State Coal Dependence vs. Mean RE Support", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "fossil_re_scatter.png")
plt.show()
print("Saved: fossil_re_scatter.png")

In [ ]:
# Figure 4: RE support by ideology group 
# Bin ideology into 3 groups for visualization
df["ideology_group"] = pd.cut(
    df["ideology"],
    bins=[0, 3, 5, 7],
    labels=["Liberal", "Moderate", "Conservative"]
)

group_means = df.groupby("ideology_group")["re_support"].mean()
group_sems  = df.groupby("ideology_group")["re_support"].sem()

fig, ax = plt.subplots(figsize=(6, 4))
colors = [GREEN, "#7CB98F", "#C8E6C9"]
bars = ax.bar(group_means.index, group_means.values,
              yerr=group_sems.values, capsize=4,
              color=colors, edgecolor="white")
ax.set_ylim(0, 5)
ax.set_ylabel("Mean RE Support (1–5)", fontsize=11)
ax.set_title("RE Support by Political Ideology", fontsize=13, fontweight="bold")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_OUT / "re_support_by_ideology.png")
plt.show()
print("Saved: re_support_by_ideology.png")

In [ ]:
# Figure 5: Correlation heatmap 
corr_cols = [c for c in ["re_support", "ideology", "climate_concern",
                          "education", "income", "age", "coal_share"] if c in df.columns]
corr = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn",
            center=0, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Matrix: Key Variables", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(FIG_OUT / "correlation_heatmap.png")
plt.show()
print("Saved: correlation_heatmap.png")